# VoxClone — XTTS v2 Server di Google Colab

## Urutan Menjalankan
| Langkah | Cell | Keterangan |
|---------|------|------------|
| 1 | Cell 1 | Cek GPU + Mount Drive |
| 2 | Cell 2 | Install + Patch (lalu **Restart session**) |
| 3 | Cell 1 | Jalankan lagi setelah restart |
| 4 | Cell 3 | Jalankan server XTTS |

Setelah Cell 3 selesai, update file `.env` di backend Go dengan kedua URL yang muncul.

In [ ]:
# =====================================================================
# Cell 1: Cek GPU & Mount Drive
# =====================================================================
from google.colab import drive
import os, torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU tidak aktif! Runtime > Change runtime type > GPU T4")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

drive.mount('/content/drive', force_remount=True)
gdrive_cache = "/content/drive/MyDrive/XTTS_Model_Cache"
os.makedirs(gdrive_cache, exist_ok=True)
os.environ["COQUI_TTS_HOME"] = gdrive_cache
os.environ["MPLBACKEND"] = "Agg"
print(f"✅ Cache model: {gdrive_cache}")

In [ ]:
# =====================================================================
# Cell 2: Install Dependencies + Patch
# Jalankan sekali per sesi, lalu Restart session
# =====================================================================
import subprocess, pathlib, inspect

# --- Downgrade PyTorch ke versi kompatibel XTTS ---
print("-> Uninstall PyTorch lama...")
subprocess.run(["pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
               capture_output=True)

print("-> Install PyTorch 2.4.1 (cu124)...")
subprocess.run([
    "pip", "install",
    "torch==2.4.1", "torchvision==0.19.1", "torchaudio==2.4.1",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)
print("✅ PyTorch 2.4.1 terinstal.")

# --- Install dependency sistem ---
print("-> Install portaudio...")
subprocess.run(["apt-get", "install", "-qq", "-y", "portaudio19-dev"], check=True)
print("✅ Portaudio terinstal.")

# --- Install XTTS & localtunnel ---
print("-> Install xtts-api-server...")
result = subprocess.run(
    ["pip", "install", "xtts-api-server"],
    capture_output=False
)
if result.returncode != 0:
    print("❌ Gagal install xtts-api-server.")
    raise SystemExit(1)

subprocess.run(["npm", "install", "-g", "localtunnel"],
               check=True, capture_output=True)
print("✅ XTTS & localtunnel terinstal.")

# --- Patch Coqpit bug 1: issubclass tidak support generic type Python 3.10+ ---
print("-> Patch Coqpit (fix 1)...")
import coqpit.coqpit as _cq
coqpit_path = pathlib.Path(inspect.getfile(_cq))
source = coqpit_path.read_text()

if "_COQPIT_PATCHED_" not in source:
    new = source.replace(
        "    if issubclass(field_type, Serializable):",
        "    if isinstance(field_type, type) and issubclass(field_type, Serializable):  # _COQPIT_PATCHED_"
    )
    coqpit_path.write_text(new)
    print("✅ Patch 1 diterapkan.")
else:
    print("✅ Patch 1 sudah ada, skip.")

# --- Patch Coqpit bug 2: union type Python 3.10+ (X | Y) ---
print("-> Patch Coqpit (fix 2)...")
source = coqpit_path.read_text()

if "_COQPIT_PATCHED_UNION_" not in source:
    OLD = '    raise ValueError(f" [!] \'{type(x)}\' value type of \'{x}\' does not match \'{field_type}\' field type.")'
    NEW = '''    import types as _t
    if isinstance(field_type, _t.UnionType):  # _COQPIT_PATCHED_UNION_
        for arg in field_type.__args__:
            try:
                return _deserialize(x, arg)
            except (ValueError, TypeError):
                continue
        return x
    raise ValueError(f" [!] '{type(x)}' value type of '{x}' does not match '{field_type}' field type.")'''
    coqpit_path.write_text(source.replace(OLD, NEW, 1))
    print("✅ Patch 2 diterapkan.")
else:
    print("✅ Patch 2 sudah ada, skip.")

print("\n✅ Semua selesai! Sekarang: Runtime > Restart session")

In [ ]:
# =====================================================================
# Cell 3: Jalankan Server XTTS + Upload Server + Tunnel
# Jalankan setelah Cell 1 (post-restart)
# =====================================================================
import subprocess, time, re, os, urllib.request
from pathlib import Path

PORT = 5002
UPLOAD_PORT = 5003
GDRIVE_SPEAKER_DIR = "/content/drive/MyDrive/XTTS_Speakers"
SPEAKER_DIR = "/tmp/xtts_speakers"

# Buat symlink dari /tmp ke Google Drive agar file speaker permanen
os.makedirs(GDRIVE_SPEAKER_DIR, exist_ok=True)
if not os.path.exists(SPEAKER_DIR):
    os.symlink(GDRIVE_SPEAKER_DIR, SPEAKER_DIR)
    print(f"✅ Symlink: {SPEAKER_DIR} → {GDRIVE_SPEAKER_DIR}")
else:
    print("✅ SPEAKER_DIR sudah ada.")

# --- Buat mini upload server (port 5003) ---
upload_server_code = f"""
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import shutil, os, uvicorn

app = FastAPI()
SPEAKER_DIR = "{SPEAKER_DIR}"

@app.post("/upload_speaker/")
async def upload_speaker(file: UploadFile = File(...)):
    dest = os.path.join(SPEAKER_DIR, file.filename)
    with open(dest, "wb") as f:
        shutil.copyfileobj(file.file, f)
    return JSONResponse({{"speaker_path": dest, "filename": file.filename}})

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port={UPLOAD_PORT})
"""

with open("/tmp/upload_server.py", "w") as f:
    f.write(upload_server_code)

log_file = open("/tmp/xtts_server.log", "w")

# --- Jalankan XTTS server ---
print("-> Menjalankan XTTS server...")
MODEL_DIR = "/content/drive/MyDrive/XTTS_Model_Cache"
os.makedirs(MODEL_DIR, exist_ok=True)

server_proc = subprocess.Popen(
    ["python", "-m", "xtts_api_server", "--port", str(PORT), "--device", "cuda",
     "-sf", SPEAKER_DIR, "-mf", MODEL_DIR],
    stdout=log_file, stderr=log_file
)

# --- Jalankan upload server ---
print("-> Menjalankan upload server di port 5003...")
upload_proc = subprocess.Popen(
    ["python", "/tmp/upload_server.py"],
    stdout=log_file, stderr=log_file
)

# --- Tunggu XTTS server siap ---
print("-> Menunggu server siap.", end="", flush=True)
for _ in range(300):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/docs", timeout=2)
        print("\n✅ XTTS server siap!")
        break
    except:
        print(".", end="", flush=True)
        time.sleep(1)
else:
    print("\n❌ Timeout. Cek log: !tail -n 50 /tmp/xtts_server.log")

# --- Buat tunnel untuk kedua port ---
print("-> Membuat tunnel untuk port 5002 (XTTS)...")
tunnel_xtts = subprocess.Popen(
    ["npx", "localtunnel", "--port", str(PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

print("-> Membuat tunnel untuk port 5003 (Upload)...")
tunnel_upload = subprocess.Popen(
    ["npx", "localtunnel", "--port", str(UPLOAD_PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

def get_url(proc):
    for _ in range(30):
        line = proc.stdout.readline()
        match = re.search(r'https://[a-z0-9\-]+\.loca\.lt', line)
        if match:
            return match.group(0)
        time.sleep(1)
    return None

xtts_url = get_url(tunnel_xtts)
upload_url = get_url(tunnel_upload)
ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()

print("\n" + "="*55)
print(f"🎙️  XTTS URL   : {xtts_url}")
print(f"📤  Upload URL : {upload_url}")
print(f"🔑  Password   : {ip}")
print("="*55)
print("\nUpdate file .env di backend Go:")
print(f"   XTTS_SERVER_URL={xtts_url}")
print(f"   XTTS_UPLOAD_URL={upload_url}")

## Cell Debug (Opsional)
Jalankan cell di bawah hanya jika ada masalah.

In [ ]:
# =====================================================================
# Debug 1: Cek log server terbaru
# =====================================================================
!tail -n 50 /tmp/xtts_server.log

In [ ]:
# =====================================================================
# Debug 2: Cek status server
# =====================================================================
import subprocess, urllib.request, json

# Cek port aktif
result = subprocess.run(["fuser", "5002/tcp", "5003/tcp"], capture_output=True, text=True)
print("Port aktif:", result.stdout or "Tidak ada")

# Cek XTTS server
try:
    res = urllib.request.urlopen("http://localhost:5002/languages")
    print("✅ XTTS server hidup")
    print("Languages:", json.loads(res.read()))
except:
    print("❌ XTTS server mati")

# Cek upload server
try:
    urllib.request.urlopen("http://localhost:5003/docs", timeout=3)
    print("✅ Upload server hidup")
except:
    print("❌ Upload server mati")

In [ ]:
# =====================================================================
# Debug 3: Cek file speaker di Google Drive
# =====================================================================
import os
speaker_dir = "/content/drive/MyDrive/XTTS_Speakers"
files = os.listdir(speaker_dir)
print(f"Total speaker: {len(files)}")
for f in files:
    size = os.path.getsize(os.path.join(speaker_dir, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
# =====================================================================
# Debug 4: Cek GPU usage
# =====================================================================
import subprocess
subprocess.run(["nvidia-smi"])

In [ ]:
# =====================================================================
# Debug 5: Bebaskan port (gunakan jika Cell 3 error 'address already in use')
# =====================================================================
import subprocess
subprocess.run(["fuser", "-k", "5002/tcp", "5003/tcp"], capture_output=True)
print("✅ Port 5002 & 5003 dibebaskan. Jalankan Cell 3 lagi.")

In [ ]:
# =====================================================================
# Keep Alive: Jalankan setelah Cell 3 agar server tidak idle
# =====================================================================
import time, threading, urllib.request

def keep_alive():
    while True:
        time.sleep(300)  # ping tiap 5 menit
        try:
            urllib.request.urlopen("http://localhost:5002/languages")
            print(".", end="", flush=True)
        except:
            pass

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("✅ Keep-alive aktif (ping tiap 5 menit).")